# Flight Delay Prediction
## Regression Engine to predict Arrival Delay

This notebook contains the code to train a regression engine which predicts the Arrival delay period (in minutes) for delayed flights.

In [6]:
# Pre-requisites
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Store the classifier models to save time
import joblib

# Preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Classifiers from scikit-learn
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import GradientBoostingRegressor

# Performance metrics
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

In [2]:
# df = pd.read_csv("Data/flight_and_weather.csv", index_col=0)
# print(f"\nShape: {df.shape}", end="\n\n")
# df.info()


Shape: (1837937, 33)

<class 'pandas.core.frame.DataFrame'>
Index: 1837937 entries, 0 to 1837936
Data columns (total 33 columns):
 #   Column           Dtype  
---  ------           -----  
 0   Year             int64  
 1   Quarter          int64  
 2   Month            int64  
 3   DayofMonth       int64  
 4   FlightDate       object 
 5   OriginAirportID  int64  
 6   Origin           object 
 7   DestAirportID    int64  
 8   Dest             object 
 9   DepTime          float64
 10  DepDelayMinutes  float64
 11  DepDel15         float64
 12  ArrTime          float64
 13  ArrDelayMinutes  float64
 14  ArrDel15         float64
 15  Time_new         int64  
 16  CRSDepTime       int64  
 17  CRSArrTime       int64  
 18  airport          object 
 19  date             object 
 20  tempF            float64
 21  WindChillF       float64
 22  humidity         float64
 23  windspeedKmph    float64
 24  WindGustKmph     float64
 25  winddirDegree    float64
 26  weatherCode      int64  

In [7]:
df = pd.read_csv("Data/flight_and_weather.csv", index_col=0)
print(f"\nShape: {df.shape}", end="\n\n")
df.info()


Shape: (1837937, 33)

<class 'pandas.core.frame.DataFrame'>
Index: 1837937 entries, 0 to 1837936
Data columns (total 33 columns):
 #   Column           Dtype  
---  ------           -----  
 0   Year             int64  
 1   Quarter          int64  
 2   Month            int64  
 3   DayofMonth       int64  
 4   FlightDate       object 
 5   OriginAirportID  int64  
 6   Origin           object 
 7   DestAirportID    int64  
 8   Dest             object 
 9   DepTime          float64
 10  DepDelayMinutes  float64
 11  DepDel15         float64
 12  ArrTime          float64
 13  ArrDelayMinutes  float64
 14  ArrDel15         float64
 15  Time_new         int64  
 16  CRSDepTime       int64  
 17  CRSArrTime       int64  
 18  airport          object 
 19  date             object 
 20  tempF            float64
 21  WindChillF       float64
 22  humidity         float64
 23  windspeedKmph    float64
 24  WindGustKmph     float64
 25  winddirDegree    float64
 26  weatherCode      int64  

### Eliminating Redundancy
|Column|Reason for elimination|
|:-|:-|
|FlightDate| The columns Year, Month and DayofMonth give this information in separate columns|
|OriginAirportID| It gives the same information as Origin|
|DestAirportID| It gives the same information as Dest|
|**CRSArrTime, ArrTime and ArrDel15**| They **leak information about target ArrDelayMinutes**|
|Time_new|It is a duplicate of time|
|date |It is a duplicate of FlightDate|
|airport|It is a duplicate of Origin|

In [ ]:
# # Dropping columns with redundant or duplicate data
# df.drop(columns=["FlightDate",
#                  "OriginAirportID",
#                  "DestAirportID",
#                  "CRSArrTime",
#                  "ArrTime",
#                  "ArrDel15",
#                  "Time_new",
#                  "date",
#                  "airport"],
#         inplace=True)
# print(f"\nShape: {df.shape}", end="\n\n")
# print(df.info())

# # df.to_csv("./Data/flight_and_weather_without_redundant_info.csv")


Shape: (1837937, 24)

<class 'pandas.core.frame.DataFrame'>
Index: 1837937 entries, 0 to 1837936
Data columns (total 24 columns):
 #   Column           Dtype  
---  ------           -----  
 0   Year             int64  
 1   Quarter          int64  
 2   Month            int64  
 3   DayofMonth       int64  
 4   Origin           object 
 5   Dest             object 
 6   DepTime          float64
 7   DepDelayMinutes  float64
 8   DepDel15         float64
 9   ArrDelayMinutes  float64
 10  CRSDepTime       int64  
 11  tempF            float64
 12  WindChillF       float64
 13  humidity         float64
 14  windspeedKmph    float64
 15  WindGustKmph     float64
 16  winddirDegree    float64
 17  weatherCode      int64  
 18  precipMM         float64
 19  visibility       float64
 20  pressure         float64
 21  cloudcover       float64
 22  DewPointF        float64
 23  time             int64  
dtypes: float64(15), int64(7), object(2)
memory usage: 350.6+ MB
None


In [8]:
# Dropping columns with redundant or duplicate data
df.drop(columns=["FlightDate",
                 "OriginAirportID",
                 "DestAirportID",
                 "CRSArrTime",
                 "ArrTime",
                 "ArrDel15",
                 "Time_new",
                 "date",
                 "airport"],
        inplace=True)
print(f"\nShape: {df.shape}", end="\n\n")
print(df.info())
 
# Save already before in 05_Classification.ipynb
# df.to_csv("./Data/flight_and_weather_without_redundant_info.csv")


Shape: (1837937, 24)

<class 'pandas.core.frame.DataFrame'>
Index: 1837937 entries, 0 to 1837936
Data columns (total 24 columns):
 #   Column           Dtype  
---  ------           -----  
 0   Year             int64  
 1   Quarter          int64  
 2   Month            int64  
 3   DayofMonth       int64  
 4   Origin           object 
 5   Dest             object 
 6   DepTime          float64
 7   DepDelayMinutes  float64
 8   DepDel15         float64
 9   ArrDelayMinutes  float64
 10  CRSDepTime       int64  
 11  tempF            float64
 12  WindChillF       float64
 13  humidity         float64
 14  windspeedKmph    float64
 15  WindGustKmph     float64
 16  winddirDegree    float64
 17  weatherCode      int64  
 18  precipMM         float64
 19  visibility       float64
 20  pressure         float64
 21  cloudcover       float64
 22  DewPointF        float64
 23  time             int64  
dtypes: float64(15), int64(7), object(2)
memory usage: 350.6+ MB
None


In [ ]:
# labelEncoder = LabelEncoder()
# df["Origin"] = labelEncoder.fit_transform(df["Origin"])
# df["Dest"] = labelEncoder.fit_transform(df["Dest"])
# # Only need the observations where the flight is delayed
# df = df[df["ArrDelayMinutes"] > 0]
# df.reset_index(inplace=True, drop=True)
# # print(df.columns)
# # print(df.shape)
# features = df.loc[:, df.columns != "ArrDelayMinutes"]
# labels = np.asarray(df["ArrDelayMinutes"])
# # print(features.shape)
# # print(labels.shape)
# # df.to_csv("Data/flight_and_weather_encoded_regression.csv")

In [10]:
labelEncoder = LabelEncoder()
df["Origin"] = labelEncoder.fit_transform(df["Origin"])
df["Dest"] = labelEncoder.fit_transform(df["Dest"])
# Only need the observations where the flight is delayed
df = df[df["ArrDelayMinutes"] > 0]
df.reset_index(inplace=True, drop=True)
# print(df.columns)
# print(df.shape)
features = df.loc[:, df.columns != "ArrDelayMinutes"]
labels = np.asarray(df["ArrDelayMinutes"])
print(features.shape)
print(labels.shape)
# df.to_csv("Data/flight_and_weather_encoded_regression.csv")

(731205, 23)
(731205,)


In [ ]:
# # Number of samples/observations/rows is greater than 100,000
# print(f"\nDataset shape: {df.shape}")
# features_train, features_test, labels_train, labels_test = train_test_split(features, labels, test_size=0.20, random_state=42)
# print(f"features_train shape: {features_train.shape} | features_test shape: {features_test.shape}")
# print(f"labels_train shape: {labels_train.shape} | labels_test shape: {labels_test.shape}")
# # print(f"{features_train.shape[1]} Features: {features_train.columns.to_list()}")
# del df


Dataset shape: (700439, 24)
features_train shape: (560351, 23) | features_test shape: (140088, 23)
labels_train shape: (560351,) | labels_test shape: (140088,)


In [11]:
# Number of samples/observations/rows is greater than 100,000
print(f"\nDataset shape: {df.shape}")
features_train, features_test, labels_train, labels_test = train_test_split(features, labels, test_size=0.20, random_state=42)
print(f"features_train shape: {features_train.shape} | features_test shape: {features_test.shape}")
print(f"labels_train shape: {labels_train.shape} | labels_test shape: {labels_test.shape}")
# print(f"{features_train.shape[1]} Features: {features_train.columns.to_list()}")
del df


Dataset shape: (731205, 24)
features_train shape: (584964, 23) | features_test shape: (146241, 23)
labels_train shape: (584964,) | labels_test shape: (146241,)


In [24]:
# بعد الـ cell الخاص بـ train_test_split
# اطبع أسماء الأعمدة التي دخلت للموديل فعلياً
print("=== الأعمدة التي دخلت للموديل ===")
print(f"عدد الأعمدة: {features_train.shape[1]}")
print()
print(features_train.columns.tolist())

=== الأعمدة التي دخلت للموديل ===
عدد الأعمدة: 23

['Year', 'Quarter', 'Month', 'DayofMonth', 'Origin', 'Dest', 'DepTime', 'DepDelayMinutes', 'DepDel15', 'CRSDepTime', 'tempF', 'WindChillF', 'humidity', 'windspeedKmph', 'WindGustKmph', 'winddirDegree', 'weatherCode', 'precipMM', 'visibility', 'pressure', 'cloudcover', 'DewPointF', 'time']


In [28]:
features_train.head()

,Year,Quarter,Month,DayofMonth,Origin,Dest,DepTime,DepDelayMinutes,DepDel15,CRSDepTime,...,windspeedKmph,WindGustKmph,winddirDegree,weatherCode,precipMM,visibility,pressure,cloudcover,DewPointF,time
528069,2025,1,3,20,2,4,818.0,27.0,1.0,751,...,5.5,40.027097,55.0,1,0.0,16.09,1019.4,0.0,20.30,800
655488,2025,3,7,26,11,13,2014.0,15.0,1.0,1959,...,11.2,53.708000,160.0,8,0.0,16.09,1014.3,75.0,71.06,2000
635300,2025,2,6,12,0,2,2331.0,217.0,1.0,1954,...,13.0,26.756800,160.0,7,0.0,16.09,1017.0,50.0,69.98,2000
176373,2024,2,4,19,0,7,2207.0,2.0,0.0,2205,...,22.3,27.948364,230.0,3,0.0,16.09,1014.6,75.0,61.70,2200
265636,2024,2,6,8,5,9,2059.0,39.0,1.0,2020,...,22.3,25.949045,150.0,2,0.0,16.09,1011.2,0.0,69.08,2000


In [ ]:
# perf_df = pd.DataFrame(columns=["Regressors", "MSE", "RMSE", "MAE", "R2"])
# def print_metrics(labels_test, model_pred, regressor_name, perf_df):
    
#     mse = mean_squared_error(labels_test, model_pred)
#     rmse = np.sqrt(mse)
#     mae = mean_absolute_error(labels_test, model_pred)
#     r2 = r2_score(labels_test, model_pred)
    
#     print(f"MSE      : {mse}", end="\n\n")
#     print(f"RMSE     : {rmse}", end="\n\n")
#     print(f"MAE      : {mae}", end="\n\n")
#     print(f"R2 Score : {r2}", end="\n\n")
    
#     perf_df = perf_df.append({"Regressors": regressor_name,
#                                         "MSE": mse, 
#                                         "RMSE": rmse,
#                                         "MAE": mae,
#                                         "R2": r2}, ignore_index=True)
#     return perf_df

In [12]:
perf_df = pd.DataFrame(columns=["Regressors", "MSE", "RMSE", "MAE", "R2"])

def print_metrics(labels_test, model_pred, regressor_name, perf_df):
    
    mse = mean_squared_error(labels_test, model_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(labels_test, model_pred)
    r2 = r2_score(labels_test, model_pred)
    
    print(f"MSE      : {mse}\n")
    print(f"RMSE     : {rmse}\n")
    print(f"MAE      : {mae}\n")
    print(f"R2 Score : {r2}\n")
    
    perf_df = pd.concat([perf_df, pd.DataFrame([{
        "Regressors": regressor_name,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    }])], ignore_index=True)
    
    return perf_df

## Training Different Regression Models

### Linear Regression

In [ ]:
# # model = LinearRegression(n_jobs=-1)
# # model.fit(features_train, labels_train)
# # joblib.dump(model, "./Regressors/LogisticRegression.joblib")
# model = joblib.load("./Regressors/LogisticRegression.joblib")
# model_pred = model.predict(features_test)
# perf_df = print_metrics(labels_test, model_pred, "LinearRegression", perf_df)
# del model
# del model_pred

MSE      : 246.11533169589046

RMSE     : 15.688063350710006

MAE      : 10.633112849932374

R2 Score : 0.9312750617189874



In [15]:
# model = LinearRegression(n_jobs=-1)
# model.fit(features_train, labels_train)
# joblib.dump(model, "./Regressors/LogisticRegression.joblib")
model = joblib.load("./Regressors/LogisticRegression.joblib")
model_pred = model.predict(features_test)
perf_df = print_metrics(labels_test, model_pred, "LinearRegression", perf_df)
del model
del model_pred

MSE      : 300.40108966827506

RMSE     : 17.332082669669997

MAE      : 11.737598923992808

R2 Score : 0.9582691411225007



### Decision Tree Regressor

In [ ]:
# # model = DecisionTreeRegressor()
# # model.fit(features_train, labels_train)
# # joblib.dump(model, "./Regressors/DecisionTreeRegressor.joblib")
# model = joblib.load("./Regressors/DecisionTreeRegressor.joblib")
# model_pred = model.predict(features_test)
# perf_df = print_metrics(labels_test, model_pred, "DecisionTreeRegressor", perf_df)
# del model
# del model_pred

MSE      : 472.70934341271203

RMSE     : 21.741879942008513

MAE      : 14.630232425332649

R2 Score : 0.8680012324829942



In [16]:
# model = DecisionTreeRegressor()
# model.fit(features_train, labels_train)
# joblib.dump(model, "./Regressors/DecisionTreeRegressor.joblib")
model = joblib.load("./Regressors/DecisionTreeRegressor.joblib")
model_pred = model.predict(features_test)
perf_df = print_metrics(labels_test, model_pred, "DecisionTreeRegressor", perf_df)
del model
del model_pred

MSE      : 591.02282533626

RMSE     : 24.310961012190777

MAE      : 16.232677566482725

R2 Score : 0.9178968020897526



### XGBoost

In [ ]:
# # model = GradientBoostingRegressor()
# # model.fit(features_train, labels_train)
# # joblib.dump(model, "./Regressors/GradientBoostingRegressor.joblib")
# model = joblib.load("./Regressors/GradientBoostingRegressor.joblib")
# model_pred = model.predict(features_test)
# perf_df = print_metrics(labels_test, model_pred, "GradientBoostingRegressor", perf_df)
# del model
# del model_pred

MSE      : 234.03676085446193

RMSE     : 15.298260059708161

MAE      : 10.367285945784936

R2 Score : 0.9346478667770067



In [17]:
# model = GradientBoostingRegressor()
# model.fit(features_train, labels_train)
# joblib.dump(model, "./Regressors/GradientBoostingRegressor.joblib")
model = joblib.load("./Regressors/GradientBoostingRegressor.joblib")
model_pred = model.predict(features_test)
perf_df = print_metrics(labels_test, model_pred, "GradientBoostingRegressor", perf_df)
del model
del model_pred

MSE      : 287.1414589746095

RMSE     : 16.94524886139503

MAE      : 11.476553146117451

R2 Score : 0.9601111310362396



### Random Forest

In [ ]:
# # model = RandomForestRegressor(n_jobs=-1)
# # model.fit(features_train, labels_train)
# # joblib.dump(model, "./Regressors/RandomForestRegressor.joblib")
# model = joblib.load("./Regressors/RandomForestRegressor.joblib")
# model_pred = model.predict(features_test)
# perf_df = print_metrics(labels_test, model_pred, "RandomForestRegressor", perf_df)
# del model
# del model_pred

MSE      : 228.859378607919

RMSE     : 15.128098975347795

MAE      : 10.419297918179321

R2 Score : 0.9360935925385799



In [18]:
# model = RandomForestRegressor(n_jobs=-1)
# model.fit(features_train, labels_train)
# joblib.dump(model, "./Regressors/RandomForestRegressor.joblib")
model = joblib.load("./Regressors/RandomForestRegressor.joblib")
model_pred = model.predict(features_test)
perf_df = print_metrics(labels_test, model_pred, "RandomForestRegressor", perf_df)
del model
del model_pred

MSE      : 283.1838378601931

RMSE     : 16.828066967426565

MAE      : 11.59984760907278

R2 Score : 0.9606609124248447



### Extra Trees Regressor

In [ ]:
# model = ExtraTreesRegressor(n_jobs=-1)
# model.fit(features_train, labels_train)
# # joblib.dump(model, "./Regressors/ExtraTreesRegressor.joblib")
# # model = joblib.load("./Regressors/ExtraTreesRegressor.joblib")
# model_pred = model.predict(features_test)
# perf_df = print_metrics(labels_test, model_pred, "ExtraTreesRegressor", perf_df)

MSE      : 232.2070527990739

RMSE     : 15.238341537026722

MAE      : 10.500640740106219

R2 Score : 0.9351587921724802



In [13]:
# model = ExtraTreesRegressor(n_jobs=-1)
# model.fit(features_train, labels_train)
# joblib.dump(model, "./Regressors/ExtraTreesRegressor.joblib")
model = joblib.load("./Regressors/ExtraTreesRegressor.joblib")
model_pred = model.predict(features_test)
perf_df = print_metrics(labels_test, model_pred, "ExtraTreesRegressor", perf_df)

MSE      : 290.79963794558296

RMSE     : 17.052848382178944

MAE      : 11.717879732769878

R2 Score : 0.9596029472924492



C:\Users\DELL\AppData\Local\Temp\ipykernel_13392\3120733534.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  perf_df = pd.concat([perf_df, pd.DataFrame([{


## Performance Summary

In [ ]:
# # Set name of the regressors as index labels
# perf_df.set_index("Regressors", inplace=True)
# perf_df

,MSE,RMSE,MAE,R2
Regressors,,,,
LinearRegression,246.115332,15.688063,10.633113,0.931275
DecisionTreeRegressor,472.709343,21.741880,14.630232,0.868001
GradientBoostingRegressor,234.036761,15.298260,10.367286,0.934648
RandomForestRegressor,228.859379,15.128099,10.419298,0.936094
ExtraTreesRegressor,232.207053,15.238342,10.500641,0.935159


In [19]:
# Set name of the regressors as index labels
perf_df.set_index("Regressors", inplace=True)
perf_df

,MSE,RMSE,MAE,R2
Regressors,,,,
NaN,290.799638,17.052848,11.717880,0.959603
LinearRegression,300.401090,17.332083,11.737599,0.958269
DecisionTreeRegressor,591.022825,24.310961,16.232678,0.917897
GradientBoostingRegressor,287.141459,16.945249,11.476553,0.960111
RandomForestRegressor,283.183838,16.828067,11.599848,0.960661
